# Complete reviewer-revision campaign

This is the recommended single entry point on the compute machine. It previews by default. Every child stage is intrinsically resumable, and rerunning this cell skips only stages with a complete summary.

<!-- reviewer-resume-contract -->
## Execution and resume contract

This notebook is aligned with the reviewer-revision implementation. Expensive work is checkpointed and safe to restart with the same configuration. Do not change methods, seeds, thresholds, or output paths while resuming. Saved outputs remain provisional until the compute-machine run and verification gates complete.


In [ ]:
from pathlib import Path
import os, shlex, subprocess, sys
def find_package_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / 'calcium-transient-rising-flank'):
            if (candidate / 'pyproject.toml').is_file() and (candidate / 'examples' / 'run_revision_campaign.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate the calcium-transient-rising-flank package root')

PACKAGE_ROOT = find_package_root()
RUNNER_PYTHON = next((str(path) for path in (PACKAGE_ROOT / '.venv/bin/python', PACKAGE_ROOT / '.venv/Scripts/python.exe') if path.is_file()), sys.executable)
RUNNER_ENV = os.environ.copy()
RUNNER_ENV['MPLBACKEND'] = 'Agg'
SOURCE_ROOT = str(PACKAGE_ROOT / 'src')
RUNNER_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, (SOURCE_ROOT, RUNNER_ENV.get('PYTHONPATH'))))
RUN_CAMPAIGN = False
SEEDS = '1,2,3,4,5,6,7,8'
DYNAMIC_N_SEEDS = 8
N_SURROGATES = 1000
OUTPUT_ROOT = PACKAGE_ROOT / 'outputs/revision_campaign'
command = [
    RUNNER_PYTHON, 'examples/run_revision_campaign.py',
    '--seeds', SEEDS, '--dynamic-n-seeds', str(DYNAMIC_N_SEEDS),
    '--n-surrogates', str(N_SURROGATES), '--output-root', str(OUTPUT_ROOT), '--resume',
]
if not RUN_CAMPAIGN:
    command.append('--dry-run')
print(shlex.join(command))
subprocess.run(command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)